# Sentinel-2 MSI RGB Composite

**Source**: `grdl/example/IO/eo/view_sentinel2.py`

## Learning Objectives

- Load Sentinel-2 L1C/L2A products
- Extract RGB bands (B04, B03, B02)
- Create true-color composite

## Data Setup

**Path Configuration**: Set path to a Sentinel-2 .SAFE directory.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

In [ ]:
# Configuration
SENTINEL2_PATH = '/path/to/your/S2A_MSIL1C_*.SAFE'
CHIP_SIZE = 2048

print(f"Sentinel-2 product: {SENTINEL2_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from grdl.IO import get_reader
from grdl.contrast import PercentileStretch

In [ ]:
# Load Sentinel-2 RGB bands
# Sentinel-2Reader auto-catalogs all bands at the target resolution when given a .SAFE directory.
# At 10m resolution: B02 (blue), B03 (green), B04 (red), B08 (NIR)
# Use bands=[0,1,2] to read B02/B03/B04 in one call.

with get_reader('sentinel2', SENTINEL2_PATH, resolution=10) as reader:
    print(f"Reader shape: {reader.get_shape()}")  # (rows, cols, n_bands)
    print(f"Available bands at 10m: {reader.metadata.bands}")
    
    # Read RGB chip: bands [0,1,2] = B02, B03, B04
    rgb = reader.read_chip(0, CHIP_SIZE, 0, CHIP_SIZE, bands=[0, 1, 2])
    
    print(f"\nRGB chip shape: {rgb.shape}")
    print(f"Data type: {rgb.dtype}")
    
    # Extract individual bands for diagnostics
    blue = rgb[:, :, 0]
    green = rgb[:, :, 1]
    red = rgb[:, :, 2]
    
    print(f"\nBand value ranges (before stretch):")
    print(f"  Blue (B02):  min={blue.min():.1f}, max={blue.max():.1f}, mean={blue.mean():.1f}")
    print(f"  Green (B03): min={green.min():.1f}, max={green.max():.1f}, mean={green.mean():.1f}")
    print(f"  Red (B04):   min={red.min():.1f}, max={red.max():.1f}, mean={red.mean():.1f}")

In [ ]:
# Apply percentile stretch to each band independently
stretch = PercentileStretch(plow=2, phigh=98)

red_stretched = stretch.apply(red)
green_stretched = stretch.apply(green)
blue_stretched = stretch.apply(blue)

# Stack RGB channels
rgb_display = np.stack([red_stretched, green_stretched, blue_stretched], axis=-1)
print(f"RGB composite shape: {rgb_display.shape}")
print(f"Display range: [{rgb_display.min():.3f}, {rgb_display.max():.3f}]")

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(rgb_display)
ax.set_title('Sentinel-2 True-Color Composite (B04/B03/B02)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cleanup memory
import gc

del rgb, red, green, blue, red_stretched, green_stretched, blue_stretched, rgb_display
gc.collect()
print("✓ Memory cleanup complete")

## Summary

**GRDL patterns**:
- ✅ `get_reader('sentinel2', path, resolution=10)` — Multi-band Sentinel-2 reader
- ✅ `.read_chip(..., bands=[0,1,2])` — Read specific bands in one call
- ✅ Returns stacked array `(rows, cols, n_bands)` for multi-band reads
- ✅ `PercentileStretch` — Per-band contrast enhancement